# Week 1 Friday - Vigenère Cryptanalysis

The main goal today is to prepare for Lab 1, which is about automatically decrypting Vigenère ciphertext.

* Please run the following libraries and functions we will need.

In [1]:
import string
import numpy as np
import textwrap
import re
from collections import Counter
from itertools import combinations
import pandas as pd
import altair as alt

english_freq = {
    'a': 0.0803,
    'b': 0.014,
    'c': 0.0232,
    'd': 0.0467,
    'e': 0.1247,
    'f': 0.0226,
    'g': 0.0209,
    'h': 0.065,
    'i': 0.0683,
    'j': 0.0012,
    'k': 0.008,
    'l': 0.0367,
    'm': 0.0255,
    'n': 0.0706,
    'o': 0.0776,
    'p': 0.0166,
    'q': 0.0011,
    'r': 0.0621,
    's': 0.0626,
    't': 0.0902,
    'u': 0.0279,
    'v': 0.0087,
    'w': 0.0236,
    'x': 0.0012,
    'y': 0.0203,
    'z': 0.0004
}


def only_letters(X, case=None):
    '''Returns the string obtained from X by removing everything but the letters.
    If case="upper" or case="lower", then the letters are all
    converted to the same case.'''
    X = ''.join(c for c in X if c in string.ascii_letters)

    if len(X) == 0:
        return None
    
    if case is None:
        return X
    elif case == "lower":
        return X.lower()
    elif case == "upper":
        return X.upper()


def string_for_code_block(X, linewidth=60):
    return '\n'.join(textwrap.wrap(X, width=linewidth))

    
def add_spaces(X, width=5, linewidth=60):
    one_line = ' '.join(textwrap.wrap(X, width=width))
    many_lines = string_for_code_block(one_line, linewidth=linewidth)
    return many_lines
    

def shift_char(ch, shift_amt):
    '''Shifts a specific character by shift_amt.
    Example:
    shift_char("Y", 3) returns "B"
    '''
    if ch in string.ascii_lowercase:
        base = 'a'
    elif ch in string.ascii_uppercase:
        base = 'A'
    # It's not clear what shifting should mean in other cases
    # so if the character is not upper or lower-case, we leave it unchanged
    else:
        return ch
    return chr((ord(ch)-ord(base)+shift_amt)%26+ord(base))


def shift_string(X, shift_amt):
    '''Shifts all characters in X by the same amount.'''
    return ''.join(shift_char(ch, shift_amt) for ch in X)


def mut_ind_co(d1, d2):
    '''For letter frequency dictionaries d1 and d2, return the Mutual Index of Coincidence.
    See Equation (5.9) on page 222 in Hoffstein, Pipher, Silverman.'''
    s = 0
    for k in d1.keys():
        s += d1.get(k, 0)*d2.get(k,0)
    return s


def ind_co(X):
    X = only_letters(X, case="upper")
    ctr = count_substrings(X, 1)
    n = sum(ctr.values())
    return (1/(n*(n-1)))*sum(f*(f-1) for f in ctr.values())


def weave(string_list):
    output = ''.join([''.join(tup) for tup in zip(*string_list)])
    # The rest is just to deal with the case of unequal string lengths
    # We assume the only possibility is that the early strings are one character longer
    last_length = len(string_list[-1])
    extra = [s[-1] for s in string_list if len(s) > last_length]
    return output + ''.join(extra)


def count_substrings(X,n):
    '''Returns a Python Counter object of all n-grams in X.'''
    if not X:
        return {}
    X = only_letters(X)
    shifts = [X[i:] for i in range(n)]
    grams = [''.join(chrs) for chrs in zip(*shifts)]
    return Counter(grams)


def get_freq(X, case="lower"):
    '''Returns the proportion that each letter occurs in "X"'''
    
    if case == "lower":
        letters = string.ascii_lowercase
    elif case == "upper":
        letters = string.ascii_uppercase
    else:
        raise ValueError("case should be 'upper' or 'lower'.")
    
    X = only_letters(X, case=case)
    n = len(X)
    ctr = count_substrings(X, 1)
    output = {}
    for char in letters:
        output[char] = ctr[char]/n
    return output


def kasiski_diffs(Y, case="upper"):
    Y = only_letters(Y, case=case)
    ctr = count_substrings(Y, 3)
    tri_reps = [k for k,v in ctr.items() if v > 1]
    diffs = []
    for tri in tri_reps:
       starts = [m.start() for m in re.finditer(f'(?={tri})', Y)]
       diffs.extend([abs(x-y) for x,y in combinations(starts, 2)])
 
    return np.array(sorted(diffs))

def freq_chart(X, case="upper"):
    '''Plot the letter frequency chart.'''
    X = only_letters(X, case=case)
    if case == "lower":
        letters = list(string.ascii_lowercase)
    elif case == "upper":
        letters = list(string.ascii_uppercase)
    else:
        raise ValueError("case should be 'upper' or 'lower'.")

    freq_dict = get_freq(X, case=case)
    ser_count = pd.Series(freq_dict, name="freq")
    df_count = pd.DataFrame(ser_count).reset_index()
    df_count.rename({"index": "letter"}, axis=1, inplace=True)

    chart = alt.Chart(df_count).mark_bar().encode(
        x=alt.X("letter", scale=alt.Scale(domain=letters)),
        y="freq",
        tooltip=["letter", "freq"]
    )

    return chart

## Vigenère ciphertext letter frequency

* Here are three Vigenère ciphertexts.  The string `Y0` was encrypted using a key length of `1`, the string `Y1` was encrypted using a key length of `5`, and the string `Y2` was encrypted using a key length of `201`.

In [3]:
Y0 = '''IWTSX UUXTW TAABP CZTNT MRWPC VTPAV DGXIW BHDAK THIWT UDAAD
LXCVS XATBB PPAXR TPCSQ DQLPC IIDHW PGTPH TRGTI ZTNUD GJHTX
CPHNB BTIGX RRXEW TGQJI IWTXG DCANB TPCHD URDBB JCXRP IXDCX
HXCHT RJGTT KTGNE XTRTD UXCUD GBPIX DCIWP IIWTN TMRWP CVTXH
DQHTG KTSQN IWTXG PSKTG HPGNT KTWDL XHXIE DHHXQ ATUDG PAXRT
PCSQD QIDHW PGTPZ TNLXI WDJIB PZXCV XIPKP XAPQA TIDTK TPIUX
GHIVA PCRTX IPEET PGHIW PIPAX RTPCS QDQUP RTPCX BEDHH XQATI
PHZXI LPHPQ GXAAX PCIXC HXVWI DUSXU UXTPC SWTAA BPCIW PIIWT
SXUUX RJAIN DUIWT SXHRG TITAD VPGXI WBEGD QATBU DGUEE GDKXS
THPED HHXQA THDAJ IXDC'''

Y1 = """SCNWB EARXA DGUFT MFNRX WXQTG FZJEZ NMRMA LNXEO DNCAX EJUEH
VDWZW HGNFF ZVUBV DVWWU NWFTG SOXLA ZMNTL DXAXM JZHYH QPBXB
MVBRF LZCKB BXRIA DMKNM SCNBK NIURF DVWLH EXXFF TIRVT SDXGB
RDWLX BPAXX UZARI HZLXH EDWYH QHJMB NICAT SOQXR DSLAT MBNBL
NWBXK UZMUR SCNBK ZYEXK RVARX UZQHP HNRMI NNBBU KZOHK ZGRVX
ZIMUH AOXLA ZMNTD DTFBM GJDMF ZFRGZ HOJOT HGJUE DOXXO DVCYB
QNCZE ZILXB SVYIX ZMBMA ZOJEB BZJGW AJKYT BZJGB LKXLL HWUXM
ZNTBM VVBTU QDUEB ZICBG RDPAM NAMBY EDNTG CCNEE LVWMA ZOCAX
CDOYB BPUMR NACAX CDBVK DONEH FVABM GHYKH AGNFY NMOII QJEBW
DNJIH RNRUE DNXEN SDXG"""

Y2 = """DRWFU VMGIO NFCHD MNCWD OJPLK IBFWE PHDMJ QNADL QAFUP LPPHZ
AYRJW WVKPD JGEUW YBVKR IUBZF HJWJT SLXUD NDCCB DMKCZ DPKKB
REPBV VJXLK XYYQV QYTEV JKUSK IQBNP DXFJB HETJB YBJHO IYCEJ
TKQKB PMMZI RFNLG QYQIR KOKCA ICQQZ VIVYI WAUNC QRMUW RTLZE
ULCWT HUKZC AQYZM DCYCP RRYGP SGETH GTYOI QWNUT BQNAE LRJGA
LRTFR UHYYK RAKTW YSXQA XINYL SYYVX ULUOU TUBMJ MMWQS PMOXO
KWXDO JWHIC VVLFF ODZLR CJDBS VYDDS ENYXR PGCSF BTCTX WQBSK
BTMLL TNKVW VEMHV RVNWR VNMEQ AWTYC WPZCE LKLYU QMHDU WXGOV
POPPA EGBAW SMCBV YLRFP CSVSW RXTFY SKQHJ UNIHR GHRXB EZBJH
ADEFS VLWLR HJXRN FCIO"""

In [5]:
# Looks like a shift cipher
# because Vigenere cipher with key length 1 is the same as a shift cipher
freq_chart(Y0)

C:\ProgramData\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

In [7]:
1/26

0.038461538461538464

We can compare `Y1` and `Y2` more directly with `alt.hconcat`, but notice that the y-axes in each diagram do not line up. To be fair, this would have been a problem even without concatenation.

In [9]:
alt.hconcat(freq_chart(Y1), freq_chart(Y2))

C:\ProgramData\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\ProgramData\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.HConcatChart(...)

Notice the difference when we use the `resolve_scale` method. Now the comparison is direct!

In [11]:
alt.hconcat(freq_chart(Y1), freq_chart(Y2)).resolve_scale(y="shared")

C:\ProgramData\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\ProgramData\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.HConcatChart(...)

In [13]:
len(only_letters(Y2))

469

Since the key for `Y2` was length 201, and the  was shifted by a different letter, the letter frequency looks much closer to uniform than `Y1`.

## Trigram frequency

* Approximately how many repeated trigrams do we expect from random text?

Consider `"SCN"` from the beginning of `Y1`.  Let's look at positions 101, 102, 103.  What's the probability that letters 101, 102, 103 are S, C, N, in that order, due to random chance?

In [16]:
1/26**3

5.689576695493855e-05

In [18]:
pow(1/26, 3)

5.689576695493856e-05

Where did 1/26^3 come from?  Each letter occurs randomly with probability 1/26, ....  Because these 3 positions are independent, the probability of all three happening is $(1/26) \cdot (1/26) \cdot (1/26)$.

In [20]:
len(Y1)

562

In [22]:
only_letters(Y1)

'SCNWBEARXADGUFTMFNRXWXQTGFZJEZNMRMALNXEODNCAXEJUEHVDWZWHGNFFZVUBVDVWWUNWFTGSOXLAZMNTLDXAXMJZHYHQPBXBMVBRFLZCKBBXRIADMKNMSCNBKNIURFDVWLHEXXFFTIRVTSDXGBRDWLXBPAXXUZARIHZLXHEDWYHQHJMBNICATSOQXRDSLATMBNBLNWBXKUZMURSCNBKZYEXKRVARXUZQHPHNRMINNBBUKZOHKZGRVXZIMUHAOXLAZMNTDDTFBMGJDMFZFRGZHOJOTHGJUEDOXXODVCYBQNCZEZILXBSVYIXZMBMAZOJEBBZJGWAJKYTBZJGBLKXLLHWUXMZNTBMVVBTUQDUEBZICBGRDPAMNAMBYEDNTGCCNEELVWMAZOCAXCDOYBBPUMRNACAXCDBVKDONEHFVABMGHYKHAGNFYNMOIIQJEBWDNJIHRNRUEDNXENSDXG'

In [24]:
len(only_letters(Y1))

469

How does the length factor into into the expected number of "SCN" repeats? Probability of a match is $1/26^3$, the total length is $469$, so the expected number of random repetitions should be $469/26^3$

In [26]:
469/(26**3)

0.02668411470186618

How many pairs of slots are there where a repetition could happen (not this particular "SCN" repetition)?

If there were only 4 slots where we could place a trigram, then we would have $\displaystyle {4 \choose 2} = 6 $ ways to arrange a repeat.  Similarly, for $n$ slots to place a trigram, we would have $\displaystyle {n \choose 2} = \dfrac{n\cdot(n-1)}{2}$ ways to arrange a repeat.

In [30]:
(469*468/2)

109746.0

In [32]:
# Rough estimate: we expect this many repetitions of trigrams due to random chance
(469*468/2) * (1/26**3)

6.244082840236686

### Kasiski Differences

The function below counts how far apart the positions are between each repeated trigram. For instance, the value 10 at index 0 implies that, if the first triagram with a repeat occurs at position $p$, then this trigram is repeated starting at position $p+10$.

In [61]:
kasiski_diffs(Y0)

array([ 10,  10,  12,  15,  15,  15,  16,  23,  23,  23,  23,  24,  26,
        29,  33,  33,  33,  42,  46,  49,  56,  56,  57,  58,  58,  65,
        65,  66,  66,  69,  69,  69,  74,  75,  75,  77,  77,  77,  77,
        77,  77,  77,  77,  77,  78,  80,  81,  87,  87,  89,  90,  90,
        90, 107, 107, 107, 107, 107, 107, 112, 112, 112, 112, 112, 112,
       119, 120, 133, 133, 134, 136, 136, 144, 147, 149, 160, 168, 170,
       171, 171, 171, 171, 171, 171, 171, 171, 174, 179, 180, 180, 180,
       180, 180, 180, 184, 184, 184, 184, 184, 184, 184, 184, 184, 186,
       187, 196, 196, 197, 197, 202, 206, 210, 211, 211, 211, 211, 211,
       211, 217, 219, 219, 219, 219, 219, 219, 226, 226, 236, 248, 252,
       261, 261, 261, 261, 261, 261, 261, 261, 261, 266, 271, 271, 276,
       277, 277, 287, 287, 292, 292, 317, 317, 318, 320, 320, 332, 332,
       346, 355, 370, 374, 374, 374, 374, 377, 377, 377, 377, 377, 380,
       396, 396, 396, 397, 397, 397, 397, 397, 397, 412, 412, 41

In [34]:
# how many repetitions of trigrams in `Y0`
len(kasiski_diffs(Y0))

182

`Y0` contains 182 repeats of trigrams.

In [36]:
# how many repetitions of trigrams in `Y2`
# This is very close to the amount we would expect due to random chance
len(kasiski_diffs(Y2))

6

In [38]:
# None of these are divisible by the key length of 201
kasiski_diffs(Y2)

array([  4,  18, 206, 284, 306, 454])

In [40]:
len(kasiski_diffs(Y1))

37

In [42]:
# Notice many of these are divisible by the key length of 5
kasiski_diffs(Y1)

array([ 10,  10,  15,  15,  15,  17,  65,  65,  75,  75,  79,  80,  90,
        90,  90, 120, 124, 154, 160, 170, 180, 180, 180, 180, 180, 180,
       198, 210, 216, 241, 254, 320, 320, 355, 370, 380, 425])

In [44]:
15%5 #15 mod 5

0

In [46]:
17%5 #17 mod 5

2

The more differences that are divisible by our guessed key length, the more evidence we have that this could potentially be our key length.

In [48]:
# how many divisible by 5?
guessed_length = 5
arr = kasiski_diffs(Y1)

t = len(arr)
s = (arr%guessed_length == 0).sum() 
#Broadcast operations across the array! 
#Each term in the array is modded by 5. 
#Checking where the elements are congruent to 0 is equivalent to checking divisibility by the guessed key length
#Sum up all of the Trues, since they count as 1.

print(s/t) # if this is a function, use `return`, not `print`

0.7837837837837838


In [50]:
# how many divisible by 7?
guessed_length = 7
arr = kasiski_diffs(Y1)

t = len(arr)
s = 0 # How many divisible by guessed_length?

#Can also do a for-loop and if statement approach. It's slower, but it's fine at this scale.
for x in arr:
    if x%guessed_length == 0: # if x is div by guessed_length
        s = s + 1

print(s/t) # if this is a function, use `return`, not `print`

0.05405405405405406


Comparing guessed lengths with less factors in common is helpful. For instance, looking for the highest proportion works better for 7-11 than it would for 6-12.  Why?  Every number divisible by 12 is automatically divisible by 6.  So this method would not quite work if the key length were 12 and we tested for divisibility by 6.

In [52]:
Y1

'SCNWB EARXA DGUFT MFNRX WXQTG FZJEZ NMRMA LNXEO DNCAX EJUEH\nVDWZW HGNFF ZVUBV DVWWU NWFTG SOXLA ZMNTL DXAXM JZHYH QPBXB\nMVBRF LZCKB BXRIA DMKNM SCNBK NIURF DVWLH EXXFF TIRVT SDXGB\nRDWLX BPAXX UZARI HZLXH EDWYH QHJMB NICAT SOQXR DSLAT MBNBL\nNWBXK UZMUR SCNBK ZYEXK RVARX UZQHP HNRMI NNBBU KZOHK ZGRVX\nZIMUH AOXLA ZMNTD DTFBM GJDMF ZFRGZ HOJOT HGJUE DOXXO DVCYB\nQNCZE ZILXB SVYIX ZMBMA ZOJEB BZJGW AJKYT BZJGB LKXLL HWUXM\nZNTBM VVBTU QDUEB ZICBG RDPAM NAMBY EDNTG CCNEE LVWMA ZOCAX\nCDOYB BPUMR NACAX CDBVK DONEH FVABM GHYKH AGNFY NMOII QJEBW\nDNJIH RNRUE DNXEN SDXG'

In [58]:
Y = only_letters(Y1)

In [62]:
Y[0::6] #every 6th letter. Y[start:end:count]. Leaving off end assumes you go to the very last element in the string

'SAURGNNCEWZVFLLJPBKASIWFTRPAXHNOLBKSYAHIKGMLDGFJUOQIYMBAZXXMQIPBGLOOMXDVYFIDNXG'

In [64]:
# Chars at position 1 mod 6
# Y[1], Y[7], Y[13], Y[19], ..
Y[1::6]

'CRFXFMXAHHVWTADZBRBDCULFSDARHQIQALUCERPNZRUADJROEDNLIABJJLMVDCAYCVCYRCOAKYQNRE'

Compare letter frequency charts for every 6th letter vs every 5th letter. The ones that match our guessed key length "look" more English, but we should get more evidence.

In [66]:
alt.hconcat(freq_chart(Y[0::6]), freq_chart(Y[1::6])).resolve_scale(y="shared")

C:\ProgramData\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\ProgramData\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.HConcatChart(...)

In [68]:
alt.hconcat(freq_chart(Y[0::5]), freq_chart(Y[1::5]), freq_chart(Y[2::5]), freq_chart(Y[3::5]), freq_chart(Y[4::5])).resolve_scale(y="shared")

C:\ProgramData\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\ProgramData\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\ProgramData\anaconda3\Lib\site-packages\altair\utils\core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
C:\ProgramData\anaconda3\Lib\site-packages\altair\utils\core.

alt.HConcatChart(...)

Recall: We can use the index of coincidence to check whether text is random or shifted English (I believe this would also work with a simple substitution, as long as all the pieces we check had been subject to the same simple subtitution key).

In [72]:
guessed_length = 6

for i in range(guessed_length):
    print(ind_co(Y[i::guessed_length]))

0.03343070431678027
0.04428904428904429
0.05627705627705628
0.056943056943056944
0.038295038295038296
0.04695304695304695


In [74]:
guessed_length = 5

for i in range(guessed_length):
    print(ind_co(Y[i::guessed_length]))

0.06680393502630977
0.06131320064058568
0.06154198123999085
0.0581102722489133
0.05586722767648434


In [78]:
# Can't use mutual index of coincidence in this part
# Looks random for these numbers
guessed_length = 5

for i in range(guessed_length):
    print(mut_ind_co(get_freq(Y[i::guessed_length]), english_freq))

0.04181702127659574
0.03527340425531914
0.03594468085106382
0.0415936170212766
0.04053763440860215


Use Mutual Index of Coincidence to try to decide if something is English.  We use the Index of Coincidence to try to decide if something is shift encrypted or substitution encrypted English.  (mut_ind_co wants "E" to be a very common letter.  ind_co wants there to be some very common letter, it doesn't care which it is)

We have the evidence to try a key of length 5!

**Exercise: Determine the key used to encrypt `Y1` and decrypt the message. This will be useful for Lab 1!**